# Daily Challenge: MCP Weather (Student)
Beginner-friendly MCP server + client (no LLMs). Complete the TODOs, then run the client.

## Setup
Run the install cell. If Colab asks, restart runtime after install.

In [ ]:
# Install MCP CLI + SDK
%pip install -qU "mcp[cli]"

In [ ]:
# Quick verify
!python --version
!mcp --help | head -n 5

## A) Server (server.py)
Implement the WeatherDemo server with one tool and one resource.

In [ ]:
%%writefile server.py
# Définition du serveur MCP de démonstration météo
# Ce serveur expose un outil pour consulter la météo et une ressource pour lister les villes disponibles.

import logging
from mcp.server.fastmcp import FastMCP

# Configuration simple des logs pour suivre les appels du serveur
logging.basicConfig(level=logging.INFO)

mcp = FastMCP("WeatherDemo")

# Base de données minimale avec quelques villes et leurs informations météo
CITY_DATA = {
    "paris": {"temp_c": 21, "condition": "sunny"},
    "london": {"temp_c": 18, "condition": "cloudy"},
    "new york": {"temp_c": 24, "condition": "breezy"},
}

@mcp.tool()
def get_weather(city: str) -> dict:
    """Retourne des informations météo pour une ville supportée."""
    # On normalise la saisie pour éviter les erreurs de casse ou d'espaces
    key = city.strip().lower()
    data = CITY_DATA.get(key)

    # Si la ville n'existe pas, on renvoie un message explicite
    if not data:
        return {
            "error": f"Unsupported city: {city}. Try one of: {', '.join(CITY_DATA)}"
        }

    logging.info("get_weather called for %s", key)
    return {"city": key, **data}

@mcp.resource("cities://list")
def list_cities() -> str:
    """Retourne la liste des villes supportées sous forme de texte."""
    # Chaque ville apparaît sur une ligne distincte pour une lecture simple
    return "\n".join(sorted(CITY_DATA))

if __name__ == "__main__":
    mcp.run()

## B) Client (client.py)
Spawn the server via STDIO, discover capabilities, and call them.

In [ ]:
%%writefile client.py
# Client MCP pour communiquer avec le serveur via STDIO
# Il initialise la session, découvre les ressources et les outils, puis les appelle.

import asyncio
import sys
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# On pointe vers le fichier serveur local pour éviter les conflits avec PATH
SERVER_PATH = Path(__file__).parent / "server.py"
server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVER_PATH)],
    env=None,
)


def extract_content(payload):
    """Extrait le texte contenu dans une réponse MCP."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        return payload.content
    return str(payload)


async def run():
    # Le client ouvre une connexion avec le serveur par stdin/stdout
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Étape obligatoire pour initialiser la conversation MCP
            await session.initialize()

            # Affichage des ressources disponibles sur le serveur
            resources = await session.list_resources()
            print("Resources:")
            for res in resources.resources:
                print(f"- {res.uri} ({res.name or ''})")

            # Affichage des outils disponibles sur le serveur
            tools = await session.list_tools()
            print("Tools:")
            for tool in tools.tools:
                print(f"- {tool.name}")

            # Lecture de la ressource exposée par le serveur
            cities = await session.read_resource("cities://list")
            print("cities://list ->")
            print(extract_content(cities))

            # Appel de l'outil météo avec une ville donnée
            weather = await session.call_tool("get_weather", {"city": "Paris"})
            print("get_weather(Paris) ->", extract_content(weather))


if __name__ == "__main__":
    asyncio.run(run())

## C) Run
Single terminal (spawns server):
```
python client.py
```
Or two terminals for debugging:
```
mcp run server.py
python client.py
```
In Colab, just run the next cell.

In [ ]:
# Exécution du client MCP pour tester l'intégration complète
# Cette commande lance le serveur et affiche les résultats du client.
!python client.py

## Troubleshooting
- `mcp: command not found` ? rerun install or restart runtime.
- No tools/resources ? check decorators, restart server.
- City missing ? use one of the supported cities listed by `cities://list`.